# Beam neutralizer

A fast beam of negative ions crosses a channel of gas. On the gas, and on the plasma it
makes itself, it changes charge state (`-`, `0`, `+`); in turn every charge change and
every ionization of the gas feeds that plasma, which a global model of the channel
balances. `Neutralizer.solve` iterates the two to a fixed point. The cross sections and
rates of `illustrative_deuterium_neutralizer` are placeholders, not data: the numbers show
what the model computes, not what a neutralizer achieves.

In [ ]:
import warnings

import plasma_plots

from wellbalanced.neutralizer import illustrative_deuterium_neutralizer

warnings.simplefilter("ignore", UserWarning)  # the placeholder-data warning
channel = illustrative_deuterium_neutralizer(points=51)
channel.beam, channel.length, channel.gas_inflow

In [ ]:
result = channel.solve()
result.converged, result.iterations, result.neutral_fraction

## The beam and its plasma

The result is a dataset: the charge-state fractions along the channel, and the channel's
plasma with every term of its balances.

In [ ]:
data = result.to_dataset()
data[["neutral_fraction", "line_density", "electron_temperature", "plasma_potential"]]

In [ ]:
with plasma_plots.figure(1, 2, figsize=(10, 3.6)) as fig:
    data.fraction.plasma.plot.profiles(x="position", over="state", ax=fig[0])
    power = data.electron_power.where(abs(data.electron_power) > 1.0, drop=True)
    fig[1].barh(power.channel.values, power.values)  # W, a channel per bar
    fig[1].set(xlabel="electron power [W]", title="electron power balance")

## The beam as volume sources

The beam's processes reach the plasma as `VolumeSource`s per unit target density, so a
model treats them with its own gas and electron densities: a global model with the mean
beam flux and fractions, a `Fluid1D` with profiles across the channel.

In [ ]:
sources = channel.beam_sources(channel.mean_flux, result.fractions.mean(axis=0))
[(s.name, s.changes, s.per, s.electron_energy) for s in sources]

## Along the channel

The global model mixes the plasma completely. The opposite limit, a plasma that does not
move along the channel, balances every axial slice in its own gas and beam:

In [ ]:
local = channel.local_plasma(result, points=11)
with plasma_plots.figure(1, 2, figsize=(10, 3.6)) as fig:
    local.density.sel(species=["e", "D2+", "D+"]).plasma.plot.profiles(
        x="position", over="species", ax=fig[0]
    )
    local.electron_temperature.plasma.plot.lineout(ax=fig[1])

A scan of the gas flow, with and without the magnetic cusps, is the
[neutralizer scan](/wellbalanced/full-examples/neutralizer-scan/) example;
`wellbalanced.cr.deuterium_neutralizer` is the same channel with OpenCR's plasma rates.